In [223]:
import sys
sys.path.insert(0, '5_uncertainty_testing/SBDynT/src')
import time
import sbdynt as sbd

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astroquery.jplsbdb import SBDB
from pathlib import Path
import math
import glob
import io

import rebound as rb
import celmech as cm
import assist
import xgboost as xgb

from linear_theory import linear_theory_prediction, make_simpler_secular_theory
from utils import ecliptic_to_icrf, icrf_to_ecliptic, ecliptic_xyz_to_elements

from tqdm import tqdm

In [224]:
ephem = assist.Ephem("data/assist/linux_m13000p17000.441", "data/assist/sb441-n16.bsp")
nesvorny_data = pd.read_csv("data/nesvorny_catalog_dataset.csv")
prediction_path = Path("data/calc_propa")
prediction_path.mkdir(parents=True, exist_ok=True)

/tmp/ipykernel_3655774/3730445920.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  nesvorny_data = pd.read_csv("data/nesvorny_catalog_dataset.csv")


# Step 1: integrating backwards

In [225]:
epoch_n = 2460200.5 # Nesvorny epoch
epoch= 2460400.5 # the epoch your data is at
conver = 58.132440867049

## 1.1 Search with Designation

In [245]:
# Get two example asteroids to test 
des = ['K09WK5U', 'K22SH4P']

In [248]:
sim = rb.Simulation()
sim.add(des[0], date = "JD%f"%epoch)
p = sim.particles[-1]
df1 = []
df1.append({
	"Des'n": des[0],
	"epoch": epoch,
	"x": p.x, 
	"y": p.y, 
	"z": p.z, 
	"vx": p.vx, 
	"vy": p.vy, 
	"vz": p.vz
})

df_e = pd.DataFrame(df1)
df_e

Searching NASA Horizons for 'K09WK5U'... 


Found: 854145 (2009 WU205) 


/home/lshen/workspace/rebound/rebound/horizons.py:184: RuntimeWarning: Warning: Mass cannot be retrieved from NASA HORIZONS. Set to 0.
  warnings.warn("Warning: Mass cannot be retrieved from NASA HORIZONS. Set to 0.", RuntimeWarning)


,Des'n,epoch,x,y,z,vx,vy,vz
0,K09WK5U,2460400.5,1.912438,-1.547085,-0.048718,0.360054,0.522457,0.025167


## 1.2 Direct input of state vector

In [247]:
df1 = []
df1.append({
	"epoch": epoch,
	"x": -1.641552,
	"y": -1.619824, 
	"z": 0.080764, 
	"vx": 0.485864, 
	"vy": -0.47166, 
	"vz": 0.019807
})

df_e = pd.DataFrame(df1)
df_e

,epoch,x,y,z,vx,vy,vz
0,2460400.5,-1.641552,-1.619824,0.080764,0.485864,-0.47166,0.019807


In [249]:
a_list = []
e_list = []
inc_list = []
Omega_list = []
omega_list = []

In [250]:
for i in range(len(df_e)):
	# Convert orbital elements in xyz, vxvyvz
	ast = df_e.iloc[i]
	sim = rb.Simulation()
	sim.add("Sun", date = "JD%f"%epoch)
	sim.add(x = ast["x"].item(),
			y = ast["y"].item(),
			z = ast["z"].item(),
			vx = ast["vx"].item()/conver,
			vy = ast["vy"].item()/conver,
			vz = ast["vz"].item()/conver,
			date = "JD%f"%epoch)
	p = sim.particles[1]
	print(p.xyz, p.vxyz)
	# Rotate to icrf
	p = ecliptic_to_icrf(p)
	print(p.xyz, p.vxyz)
	# Use ASSIST to integrate backwards to Nesvorny epoch
	sim_a = rb.Simulation()
	ex = assist.Extras(sim_a, ephem)
	ex.gr_eih_sources = 11
	sim_a.t = epoch - ephem.jd_ref
	sim_a.ri_ias15.adaptive_mode = 2
	sim_a.add(p, plane = "frame", date="JD%f"%epoch)
	# print(sim_a.particles[-2].vxyz, sim_a.particles[-1].vxyz)
	sim_a.move_to_com()
	sim_a.exit_max_distance = 50.0
	ex.integrate_or_interpolate(epoch_n - ephem.jd_ref)
	p = sim_a.particles[-1]
	p = icrf_to_ecliptic(p)
	p.vxyz = np.array(p.vxyz) * conver

	# Retrieve the orbital elements
	orbit = ecliptic_xyz_to_elements(p)
	a_list.append(orbit.a)
	e_list.append(orbit.e)
	inc_list.append(orbit.inc)
	Omega_list.append(orbit.Omega)
	omega_list.append(orbit.omega)

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
[1.9124376669620713, -1.5470848223505504, -0.04871774787785841] [0.006193688054762893, 0.008987356697778712, 0.00043291842583955244]
[1.9124376669620713, -1.4000437658246552, -0.6600926603752256] [0.006193688054762893, 0.008073533495525337, 0.003972160076478502]


/home/lshen/workspace/rebound/rebound/simulation.py:927: RuntimeWarning: Particle is being adding from a simulation that uses different units.
  warnings.warn("Particle is being adding from a simulation that uses different units.", RuntimeWarning)


In [251]:
a_list

[2.4495308709438564]

In [252]:
merged_df = pd.DataFrame({'a': a_list, 
						  'e': e_list, 
						  'Incl.': inc_list, 
						  'Node': Omega_list, 
						  'Peri.': omega_list})

merged_df

,a,e,Incl.,Node,Peri.
0,2.449531,0.075273,0.043104,-0.198593,1.245513


In [253]:
data = nesvorny_data[nesvorny_data["Des'n"] == 'K15Bz3T'][["a", "e", "Incl.", "Node", "Peri."]]
data["Incl."] = data["Incl."] * np.pi/180
data["Node"] = data["Node"] * np.pi/180
data["Peri."] = data["Peri."] * np.pi/180
data

,a,e,Incl.,Node,Peri.
901219,3.027323,0.062849,0.162224,1.730208,4.042971


# Step 2. Linear prediction

In [254]:
simpler_secular_theory = make_simpler_secular_theory()
u0_list = []
v0_list = []
g0_list = []
s0_list = []

for i in range(len(merged_df)):
	ast = merged_df.iloc[i]
	u0, v0, g0, s0 = linear_theory_prediction(ast["e"], ast['Incl.'], ast['Peri.'], ast['Node'], ast['a'], simpler_secular_theory)
	u0_list.append(u0)
	v0_list.append(v0)
	g0_list.append(g0)
	s0_list.append(s0)

merged_df['prope_h']    = [np.abs(complex(x)) for x in u0_list]
merged_df['propsini_h'] = [np.abs(complex(x)) for x in v0_list]
merged_df['g0'] = g0_list
merged_df['s0'] = s0_list

# Step 3. Predict the Proper Elements

In [255]:
merged_df["prope_h"] = np.abs(merged_df["prope_h"])
merged_df["propsini_h"] = np.abs(merged_df["propsini_h"])
merged_df['prope_h'] = pd.to_numeric(merged_df['prope_h'], errors='coerce')
merged_df['propsini_h'] = pd.to_numeric(merged_df['propsini_h'], errors='coerce')
merged_df['Node'] = pd.to_numeric(merged_df['Node'], errors='coerce')
merged_df['Peri.'] = pd.to_numeric(merged_df['Peri.'], errors='coerce')
merged_df['ecospo'] = merged_df['prope_h']*np.cos((merged_df['Node']+merged_df['Peri.']))
merged_df['esinpo'] = merged_df['prope_h']*np.sin((merged_df['Node']+merged_df['Peri.']))
merged_df['sinicosO'] = np.sin(merged_df['propsini_h']*np.pi/180)*np.cos(merged_df['Node'])
merged_df['sinisinO'] = np.sin(merged_df['propsini_h']*np.pi/180)*np.sin(merged_df['Node'])

In [256]:
features_e = ['sinicosO', 'sinisinO', 'ecospo', 'esinpo', 'a', 'g0', 'prope_h']
features_inc = ['sinicosO', 'sinisinO', 'ecospo', 'esinpo', 'a', 's0', 'propsini_h']

In [257]:
final_model_e = xgb.XGBRegressor()
final_model_e.load_model("data/models/best_model_e_final.xgb")
final_model_inc = xgb.XGBRegressor()
final_model_inc.load_model("data/models/best_model_inc_final.xgb")

In [258]:
start_t = time.process_time()
dele = final_model_e.predict(merged_df[features_e])

delsini = final_model_inc.predict(merged_df[features_inc])
eval_t = time.process_time() - start_t

merged_df["pred_e"] = merged_df["e"] + dele
merged_df["pred_sini"] = np.sin(merged_df["Incl."]) + delsini

# Step 4. Family Classification

In [259]:
def calculate_d(a_p, delta_a_p, delta_e_p, delta_sin_i_p):
	numerator = 3e4  # 3 × 10^4 m/s
	denominator = math.sqrt(a_p)
	term1 = (delta_a_p / a_p) ** 2
	term2 = 2 * (delta_e_p ** 2)
	term3 = 2 * (delta_sin_i_p ** 2)
	inside_sqrt = (5 / 4) * term1 + term2 + term3
	d = (numerator / denominator) * math.sqrt(inside_sqrt)
	return d

In [260]:
def extract_table_from_txt(path):
	with open(path, "r") as f:
		lines = f.readlines()

	start = None
	for i, line in enumerate(lines):
		if line.strip().startswith("Number"):
			start = i
			break

	end = None
	for i, line in enumerate(lines[start:], start=start):
		if line.strip().startswith("Note."):
			end = i
			break

	table_lines = lines[start:end]

	df = pd.read_csv(io.StringIO("".join(table_lines)), sep="\t+", engine="python")
	df['Name'] = df["Name"].str.replace("-","_")
	return df

# Collect all txt files in your folder
all_files = glob.glob("data/family_d_vals/*.txt")

# Extract and merge
df_list = [extract_table_from_txt(file) for file in all_files]
df_d = pd.concat(df_list, ignore_index=True)

In [261]:
df = pd.read_csv("data/merged_elements.csv")

/tmp/ipykernel_3655774/3699952119.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/merged_elements.csv")


In [ ]:
column_names = ['propa', 'prope', 'propsini', 'g', 's', 'H',
				'NumOpps', 'PackedName', 'UnpackedName']

dataset_path = Path('data/family_tables')
filenames = list(dataset_path.glob('*.csv'))

results = []

for i in range(len(merged_df)):

	ast = merged_df.iloc[i]
	ast_a = ast["a"]
	ast_prope = ast["pred_e"]
	ast_propsini = ast["pred_sini"]

	pairs = []  # (family_number, d)

	for filename in tqdm(filenames):

		df_family = pd.read_csv(filename, header=None, names=column_names)
		family_number = str(filename).split("/")[-1].split(".")[0].split("_")[1]

		desn = df_family.iloc[0, 7]
		if str(desn).isdigit():
			desn = int(desn)

		ast_f = df[df["Des'n"] == desn]

		if ast_f.empty:
			continue

		da = ast_f['a'].values[0] - ast_a
		de = ast_f['prope'].values[0] - ast_prope
		dsini = ast_f['propsini'].values[0] - ast_propsini

		d = calculate_d(ast_a, da, de, dsini)

		pairs.append((family_number, d))

	pairs_sorted = sorted(pairs, key=lambda x: x[1])

	# take top 5
	top5 = pairs_sorted[:5]

	results.append({
		"asteroid_index": i,
		"top5_families": [f for f, d in top5],
	})

results_df = pd.DataFrame(results)

100%|██████████| 153/153 [00:08<00:00, 17.87it/s]


In [273]:
dataset_path = Path('data/family_tables')
files = list(dataset_path.glob('*.csv'))

family_file_map = {
	str(f).split("/")[-1].split(".")[0].split("_")[1]: f
	for f in files
}

matches = []

for i in range(len(merged_df)):

	ast = merged_df.iloc[i]
	ast_a = ast["a"]
	ast_prope = ast["pred_e"]
	ast_propsini = ast["pred_sini"]

	top5_families = results_df.iloc[i]["top5_families"]
	matched_families = []

	for family_number in top5_families:
		if family_number not in family_file_map:
			continue

		df_family = pd.read_csv(
			family_file_map[family_number],
			header=None,
			names=column_names
		)

		row = df_d[df_d["Number"].astype(str) == str(family_number)]
		
		if row.empty:
			continue

		d_cutoff = row["HCM Cut"].values[0]

		family_matched = False

		for j in range(len(df_family)):
			fam_star = df_family.iloc[j]

			fam_match = df[df["Des'n"] == fam_star["PackedName"]]

			if fam_match.empty:
				continue

			fam_a = fam_match['a'].values[0]
			fam_e = fam_match['prope'].values[0]
			fam_s = fam_match['propsini'].values[0]

			da = fam_a - ast_a
			de = fam_e - ast_prope
			dsini = fam_s - ast_propsini

			d = calculate_d(ast_a, da, de, dsini)

			if d <= d_cutoff:
				family_matched = True
				break

		if family_matched:
			matched_families.append(family_number)

	matches.append({
		"asteroid_index": i,
		"matched_families": matched_families
	})

results_df["matched_families"] = [m["matched_families"] for m in matches]

In [281]:
# The family number the asteroid is in
print(f"The family your candidate is in numbered {results_df['matched_families'][0][0]}")

The family your candidate is in numbered 126
